# Solar Envelope (Time-Based Mode)

The **solar envelope** is the maximum buildable volume on a site that does **not** cast shadows on designated surrounding surfaces during specified hours. The concept was introduced by Ralph Knowles in *Sun, Rhythm, Form* (1981) as a zoning tool that guarantees solar access to neighbors.

## How it works

1. For each hour in the analysis period, the sun position is computed from the EPW weather file.
2. Rays are cast from every test-surface sample point toward each sun position.
3. Any voxel inside the maximum volume that is pierced by a ray is marked as shadow-casting.
4. All marked voxels are carved away. What remains is the **solar envelope**.

## When to use

Use `time-based` when you need a **hard guarantee** of direct sun access for specific dates and times:

- "No shadows on the playground between 10 AM and 2 PM on the winter solstice."
- "Guarantee morning sun on the south facade during December–February."

Unlike `benefit` or `irradiance` (which weight hours by energy), `time-based` treats every hour equally — a voxel either casts a shadow or it doesn't.


## 1. Imports and setup


In [ ]:
from pathlib import Path
from urbansolarcarver import user_config, run_pipeline
import trimesh

REPO_ROOT = Path.cwd().parent
print(f"Repository root: {REPO_ROOT}")


## 2. Input geometry

| Mesh | Purpose |
|------|---------|
| **Max volume** | The maximum buildable envelope — voxels inside will be tested and potentially carved |
| **Test surfaces** | The surfaces that must receive sun — rays are cast *from* these toward each sun position |


In [ ]:
max_volume_path = REPO_ROOT / "examples" / "test_inputs" / "full_blocks" / "maxVolume.ply"
test_surface_path = REPO_ROOT / "examples" / "test_inputs" / "full_blocks" / "testSurfaces.ply"
epw_path = str(ADDRESS_TO_EPW_FILE) #<-- ADD YOUR EPW FILE PATH HERE (donwload from ladybug.tools/epwmap/)

# Load and inspect
max_vol_mesh = trimesh.load(str(max_volume_path))
test_srf_mesh = trimesh.load(str(test_surface_path))

In [ ]:
# Gray = buildable volume, green = surfaces that need sun
max_vol_mesh.visual.face_colors = [200, 200, 200, 255]
test_srf_mesh.visual.face_colors = [0, 200, 0, 255]

scene = trimesh.Scene([max_vol_mesh, test_srf_mesh])
scene.show()


## 3. Understanding the analysis period

The analysis period defines **when** we demand solar access. It directly controls how conservative the envelope is:

- A **wider** window (more hours, more days) → more sun positions to satisfy → **smaller** envelope
- A **narrower** window → fewer constraints → **larger** envelope

We'll use **January 15, 10:00–14:00**:

| Choice | Reasoning |
|--------|-----------|
| **December 21** | Close to the winter solstice — sun is at its lowest. If the envelope is shadow-free on this date, it will be shadow-free all year (sun is higher on other days). |
| **10:00–14:00** | The 4-hour midday window when direct sun is most valuable in winter. |

> **Tip:** Start with a single winter day and a short time window for fast iteration. Widen it later.


In [ ]:
out_dir = REPO_ROOT / "examples" / "test_outputs" / "solar_envelope_demo"

cfg = user_config(
    max_volume_path   = str(max_volume_path),
    test_surface_path = str(test_surface_path),
    epw_path          = epw_path,
    out_dir           = str(out_dir),

    mode = "time-based",

    # Analysis period: December 21, 10:00–14:00
    start_month = 12, start_day = 21, start_hour = 10,
    end_month   = 12, end_day   = 21, end_hour   = 14,

    # Resolution
    voxel_size = 2.0,
    grid_step  = 2.0,
)

#prints
print(f"Mode       : {cfg.mode}")
print(f"Voxel size : {cfg.voxel_size} m")
print(f"Period     : {cfg.start_day}/{cfg.start_month} {cfg.start_hour}:00 → "
      f"{cfg.end_day}/{cfg.end_month} {cfg.end_hour}:00")


In [ ]:
#Run the USC pipeline
result = run_pipeline(cfg, cfg.out_dir)


## 4. Result

The red solid is the solar envelope. Any building massed within this shape will **not** shadow the test surfaces between 10 AM and 2 PM on Dec. 21st.

Notice how the envelope is carved on the sides facing the test surfaces — those are the directions where the low winter sun would project shadows.

If the volume is carved too aggressively try another period

In [ ]:
envelope_mesh = trimesh.load(str(result.export_path))

print("=== Solar Envelope ===")
print(f"  Vertices : {len(envelope_mesh.vertices):,}")
print(f"  Faces    : {len(envelope_mesh.faces):,}")

envelope_mesh.visual.face_colors = [220, 80, 60, 255]
trimesh.Scene([envelope_mesh]).show()


## Key parameters

| Parameter | Effect |
|-----------|--------|
| `start/end_month/day/hour` | **Analysis period.** Wider window → more conservative (smaller) envelope. |
| `voxel_size` | **Grid resolution** in metres. Start with 2–3 m for exploration, 0.5–1 m for final results. |
| `grid_step` | Surface sampling density. Must be ≤ `voxel_size`. Smaller = more accurate but slower. |
| `min_altitude` | Ignore sun below this angle (default 5°). Filters grazing sunrise/sunset rays. |


## Next steps

- **Daylight envelopes**: geometric + CIE overcast sky → Tutorial 3
- **Advanced control**: Irradiance mode with threshold and postprocess finetuning → Tutorial 4
- **Chaining modes**: feed the output of one mode into another to combine constraints → Tutorial 4
